# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection

In [2]:
# Initialize and constants

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

 20%|█████████                                    | 1/5 [00:07<00:30,  7.61s/it]/project/sandbox/projects/llm_engineering/week8/agents/deals.py:27: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  description = BeautifulSoup(description, 'html.parser').get_text()
100%|█████████████████████████████████████████████| 5/5 [00:36<00:00,  7.27s/it]


In [4]:
len(deals)

50

In [5]:
deals[44].describe()

'Title: 500W Smoke Machine 2000CFM Fog 13 Colorful LED Light for $33 + free shipping\nDetails: Clip the $5 off coupon on the page for a savings of $38. Buy Now at Walmart\nFeatures: 13 colorful LED lights wireless remote control 500W output 300ml storage capacity\nURL: https://www.dealnews.com/500-W-Smoke-Machine-2000-CFM-Fog-13-Colorful-LED-Light-for-33-free-shipping/21770239.html?iref=rss-c196'

In [6]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [7]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [8]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Refurb EcoFlow Delta 3 Plus 1024Wh Portable Power Station for $499 + free shipping
Details: Use promo code "LOVETOSAVE" for the best price we could find by $20. This is a certified refurbished item backed by a 2-year warranty from Allstate. Buy Now at eBay
Features: 
URL: https://www.dealnews.com/products/Eco-Flow/Eco-Flow-Delta-3-Plus-1024-Wh-Portable-Power-Station

In [9]:
def get_recommendations():
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection
    )
    result = completion.choices[0].message.parsed
    return result

In [10]:
result = get_recommendations()

In [11]:
len(result.deals)

5

In [12]:
result.deals[1]

Deal(product_description='The Samsung 7.5-Cubic Foot Smart Electric Dryer incorporates advanced technology with Steam Sanitize+ and Sensor Dry features, ensuring your clothes come out perfectly dry and fresh. It offers Wi-Fi connectivity for remote operation, allowing you to monitor and control your laundry from your smartphone. With a sleek design and flexible installation options, it seamlessly fits into any laundry space. This dryer comes with the added advantage of a 3-year Samsung Care+ plan, ensuring long-term protection and support for your appliance.', price=849.0, url='https://www.dealnews.com/products/Samsung/7-5-Cubic-Foot-Smart-Electric-Dryer-with-Steam-Sanitize-and-Sensor-Dry/490227.html?iref=rss-f1912')

In [13]:
from agents.scanner_agent import ScannerAgent

In [14]:
agent = ScannerAgent()
result = agent.scan()

In [15]:
result

DealSelection(deals=[Deal(product_description='The EcoFlow Delta 3 Plus is a versatile portable power station featuring a robust 1024Wh capacity, allowing you to power multiple devices at once. It is ideal for outdoor adventures, emergency backup, or even daily use at home. The unit comes certified refurbished with a two-year warranty, ensuring its reliability and performance. Whether charging phones, running appliances, or supporting medical devices, this power station delivers substantial energy conveniently.', price=499.0, url='https://www.dealnews.com/products/Eco-Flow/Eco-Flow-Delta-3-Plus-1024-Wh-Portable-Power-Station/482025.html?iref=rss-c142'), Deal(product_description='The TCL S5 58S5F is a 58-inch 4K HDR LED Smart TV packed with cutting-edge features to elevate your viewing experience. With a stunning 3840x2160p resolution, HDR PRO+ technology enhances color and contrast, making every scene lifelike. It includes built-in Fire TV with Alexa for seamless voice control and acce